# Exercício 20 — API de leitura e camada de dados

Este exercício tem três partes, na ordem da aula:

- **Parte 1 (neste notebook):** consumir uma API pública real, checar o status e normalizar a resposta.
- **Parte 3B (neste notebook):** guardar uma coleta sua de uma aula anterior em um arquivo Parquet.
- **Parte 4 (editando `api_exercicio.py`):** criar endpoints com FastAPI que respondem **lendo do Parquet**, testados pela documentação automática (`/docs`).

**Concluído quando:** a sua API local devolve dado lido do arquivo Parquet (não de uma lista no código), e você consegue explicar cada campo do JSON que ela retorna.

Copie a pasta `exercicios/` para dentro da sua pasta de entregas (`extracao-dados-trabalhos-seunome`), numa pasta `20-api-e-persistencia` dentro de `projetos/`.

## Preparação do ambiente

Como este notebook (e o `api_exercicio.py`) vão morar numa pasta própria (dentro da sua pasta de entregas, fora do material do curso), crie o ambiente e instale as dependências de novo, dentro da pasta `exercicios/` copiada:

Windows (Prompt de Comando ou Terminal integrado do VS Code):
```cmd
uv venv .venv
uv pip install -r requirements.txt
```

(o `requirements.txt` inclui `pandas` e `pyarrow` para a Parte 3B)

Mac (Terminal ou Terminal integrado do VS Code):
```bash
uv venv .venv
uv pip install -r requirements.txt
```

## Parte 0 — Escolha da API pública

Escolha uma API pública gratuita, sem necessidade de chave, para consultar na Parte 1. Sugestões (além do ViaCEP, já usado na aula):

- **https://viacep.com.br/ws/{cep}/json/**: dados de endereço a partir de um CEP (a mesma da aula, mas com outro CEP).
- **https://jsonplaceholder.typicode.com/**: API fake com posts, comentários, usuários (útil para praticar sem se preocupar com dado sensível).
- **https://api.publicapis.org/entries**: catálogo de outras APIs públicas gratuitas (uma API que lista APIs).

Registre aqui antes de começar:

**API escolhida (nome e URL do endpoint):**

> Escreva aqui.

**O que você espera que a resposta traga (quais campos, mesmo que só uma suposição antes de testar):**

> Escreva aqui.

## Parte 1 — Requisição e checagem do status

Complete a URL abaixo com o endpoint escolhido na Parte 0 e faça a requisição. **Não siga para a Parte 2 se o status não for 200:** revise a URL, a conexão, ou se a API está mesmo disponível.

In [ ]:
import requests  # já usado na Aula 8 e na primeira metade desta aula, para fazer requisições HTTP

url = ""  # cole aqui a URL escolhida na Parte 0, por exemplo "https://viacep.com.br/ws/20040020/json/"

resposta = requests.get(url)  # faz a requisição GET
print(f"Status: {resposta.status_code}")


## Parte 2 — Transformando a resposta em dicionário ou lista Python

Use `.json()` para transformar o corpo da resposta em algo que dá para manipular como dicionário ou lista Python (Seção 3 da aula).

In [ ]:
dado = resposta.json() if resposta.status_code == 200 else None  # só converte se a requisição deu certo

print(type(dado))  # confirma se virou dict ou list
dado  # mostra o conteúdo inteiro, para você inspecionar os campos disponíveis


## Parte 3 — Normalizando os campos que interessam

Olhando o resultado da Parte 2, escolha de 3 a 5 campos que interessam e organize numa lista de dicionários (mesmo que seja só um item, se a API só devolver um). Siga o modelo da Seção 4 da aula (o `enderecos_normalizados` a partir do ViaCEP).

In [ ]:
itens_normalizados = []  # lista vazia que vai guardar um dicionário "limpo" por item

# TODO: adapte o código abaixo ao formato da API que você escolheu.
# Se a resposta já for uma lista, percorra-a com um for (como na aula).
# Se a resposta for um único dicionário, monte um único item normalizado.

if dado:
    itens_normalizados.append({
        # "campo_1": dado["..."],
        # "campo_2": dado["..."],
    })

itens_normalizados


**O que observar:** confira se `itens_normalizados` não está vazio e se os campos escolhidos batem com o que você esperava na Parte 0. Se algum campo não existir na resposta (erro tipo `KeyError`), volte na Parte 2 e reveja o dicionário/lista bruto para achar o nome certo da chave.

## Parte 3B — Guardar a sua coleta em Parquet

Agora a camada de dados. Pegue uma coleta sua de uma aula anterior (o `exportacao.csv` das Aulas 4 a 13, ou a saída tratada do pipeline da Aula 10) e guarde num arquivo Parquet, para a sua API ler dele na Parte 4.

Copie o CSV da coleta para `dados/` nesta pasta e ajuste o nome do arquivo e as colunas abaixo. Se tiver uma coluna de data em texto, converta com `pd.to_datetime` antes de gravar: o Parquet vai lembrar que ela é data.

In [ ]:
import pandas as pd
from pathlib import Path

# ajuste o nome do arquivo e o separador para a sua coleta
tabela = pd.read_csv("dados/exportacao.csv", sep=";").drop_duplicates()

# complete aqui: escolha as colunas que a sua API vai servir (umas 5 a 10 bastam)
# colunas = ["id", "author", "timestamp", "likes", "plays", "hashtags"]
# registros = tabela[colunas].copy()
registros = tabela  # troque pela versão só com as colunas escolhidas

Path("dados").mkdir(exist_ok=True)
registros.to_parquet("dados/coleta.parquet", index=False)

# confira: lê o arquivo de volta e mostra o tamanho da tabela e o tipo das colunas
conferencia = pd.read_parquet("dados/coleta.parquet")
print("Arquivo dados/coleta.parquet criado.")
print(f"{len(conferencia)} linhas, {len(conferencia.columns)} colunas")
conferencia.dtypes

## Parte 4 — Endpoints que leem do Parquet

Abra `api_exercicio.py` (nesta pasta `exercicios/`) no VS Code. Ele já vem com a leitura do arquivo pronta (as funções `ler()` e `como_registros()`) e dois endpoints com `TODO`. Você completa:

1. `listar_registros(limite)`: lê o Parquet e devolve as primeiras `limite` linhas (modelo: endpoint `/posts` de `exemplos/api_dados.py`).
2. `resumo()`: lê o Parquet e usa `groupby()` para agregar por uma coluna de categoria da sua coleta (autor, hashtag, dia...) e devolver a contagem por grupo (modelo: endpoint `/autores`).

Ajuste também o `title` e a `description` do `app`.

Depois de salvar, rode o servidor num terminal, **nesta pasta `exercicios/`** (com o `dados/coleta.parquet` já criado na Parte 3B):

Windows (Prompt de Comando ou Terminal integrado do VS Code):
```cmd
uv run uvicorn api_exercicio:app --reload
```

Mac (Terminal ou Terminal integrado do VS Code):
```bash
uv run uvicorn api_exercicio:app --reload
```

E abra `http://127.0.0.1:8000/docs`.

## Parte 5 — Testando pela documentação automática

Com o servidor rodando (Parte 4) e a `/docs` aberta, teste:

1. `GET /registros` com `limite = 3`: confira que voltam só 3 linhas, com as colunas que você escolheu.
2. `GET /resumo`: confira que a contagem por grupo bate com o que você esperaria da sua coleta (compare com um `groupby()` no pandas, se quiser).
3. Pare o servidor (`Ctrl+C`), apague `dados/coleta.parquet`, suba o servidor de novo e chame `GET /resumo`: o erro `500` (com `FileNotFoundError` no terminal do servidor) confirma que a API agora depende do arquivo, não de dado embutido no código.

## Parte 6 — Registro da entrega

Crie `projetos/20-api-e-persistencia/README.md` (na sua pasta de entregas) e responda:

**API pública consultada na Parte 1 (nome e URL):**

> Escreva aqui.

**Campos extraídos na Parte 3 e o que cada um significa:**

> Escreva aqui.

**Coleta usada na Parte 3B, e quais colunas foram para o `coleta.parquet`:**

> Escreva aqui.

**Os dois endpoints da sua API (o que cada um lê do Parquet):**

> Escreva aqui.

**Exemplo de requisição e resposta** (cole a URL testada, ex.: `GET http://127.0.0.1:8000/resumo`, e o JSON devolvido, copiado da `/docs`):

> Escreva aqui.

**Como rodar o seu servidor** (o comando exato e a porta):

> Escreva aqui.

**Declaração de uso de IA:** ferramenta usada, em que trecho ou decisão, e o que você conferiu ou alterou depois (mesmo que seja "não usei IA nesta entrega").

> Escreva aqui.

## Parte 7 — Conferência final

- [ ] A Parte 0 registra a API pública escolhida e o que você esperava da resposta.
- [ ] A Parte 1 verificou o `status_code` e só seguiu porque era `200`.
- [ ] A Parte 3 mostra uma lista de dicionários com pelo menos um item normalizado.
- [ ] A Parte 3B criou `dados/coleta.parquet` a partir de uma coleta sua, e o `.parquet` está no `.gitignore`.
- [ ] `api_exercicio.py` tem os dois endpoints completos, ambos lendo do Parquet (nenhuma lista de dados fixa no código).
- [ ] Você testou os dois endpoints pela `/docs`, e testou o comportamento com o Parquet apagado.
- [ ] Você consegue explicar, para cada campo do JSON devolvido, o que ele representa.
- [ ] `projetos/20-api-e-persistencia/README.md` responde as perguntas da Parte 6, incluindo a declaração de uso de IA.
- [ ] O notebook roda do início ao fim com Kernel → Restart e Run All.
- [ ] Este notebook e o `api_exercicio.py` completo estão em `projetos/20-api-e-persistencia/` na pasta de entregas.
- [ ] Você já fez `git add`, `git commit` e `git push` dessa entrega.